In [ ]:
# ============================================================
# PROJECT 5 — HMM REGIME TRANSITION & PERSISTENCE ANALYSIS
# NIFTY 50
# ============================================================

!pip -q install yfinance hmmlearn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. DOWNLOAD NIFTY 50 DATA
# ------------------------------------------------------------
df = yf.download(
    "^NSEI",
    start="2015-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.dropna()

# ------------------------------------------------------------
# 2. FEATURES
# ------------------------------------------------------------
df["Return"] = df["Close"].pct_change()

df["Volatility"] = (
    df["Return"].rolling(20).std()
)

df["Momentum"] = (
    df["Close"] /
    df["Close"].rolling(20).mean() - 1
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

features = [
    "Return",
    "Volatility",
    "Momentum"
]

# ------------------------------------------------------------
# 3. STANDARDIZE
# ------------------------------------------------------------
scaler = StandardScaler()

X = scaler.fit_transform(
    df[features]
)

# ------------------------------------------------------------
# 4. FIT 3-STATE GAUSSIAN HMM
# ------------------------------------------------------------
model = GaussianHMM(
    n_components=3,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)

model.fit(X)

df["State"] = model.predict(X)

# ------------------------------------------------------------
# 5. REGIME LABELING
# ------------------------------------------------------------
state_stats = []

for state in range(3):

    data = df[df["State"] == state]

    state_stats.append({
        "State": state,
        "Mean Return": data["Return"].mean(),
        "Volatility": data["Return"].std()
    })

state_stats = pd.DataFrame(state_stats)

ordered_states = (
    state_stats
    .sort_values("Mean Return")
    ["State"]
    .tolist()
)

labels = [
    "Bear",
    "Neutral",
    "Bull"
]

state_mapping = {
    ordered_states[i]: labels[i]
    for i in range(3)
}

df["Regime"] = (
    df["State"]
    .map(state_mapping)
)

# ------------------------------------------------------------
# 6. TRANSITION MATRIX
# ------------------------------------------------------------
transition = model.transmat_

transition_df = pd.DataFrame(
    transition,
    index=[
        state_mapping[i]
        for i in range(3)
    ],
    columns=[
        state_mapping[i]
        for i in range(3)
    ]
)

print("=" * 70)
print("HMM REGIME TRANSITION & PERSISTENCE ANALYSIS")
print("=" * 70)

print("\nTRANSITION PROBABILITY MATRIX")
print("-" * 70)

print(
    transition_df
    .round(4)
    .to_string()
)

# ------------------------------------------------------------
# 7. REGIME PERSISTENCE
# ------------------------------------------------------------
persistence = []

for state in range(3):

    stay_probability = transition[state, state]

    switch_probability = (
        1 - stay_probability
    )

    expected_duration = (
        1 / switch_probability
        if switch_probability > 0
        else np.inf
    )

    persistence.append({
        "Regime": state_mapping[state],
        "Stay Probability": stay_probability,
        "Switch Probability": switch_probability,
        "Expected Duration (Days)": expected_duration
    })

persistence_df = pd.DataFrame(
    persistence
)

print("\nREGIME PERSISTENCE")
print("-" * 70)

print(
    persistence_df
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. OBSERVED REGIME COUNTS
# ------------------------------------------------------------
regime_counts = (
    df["Regime"]
    .value_counts()
    .reindex(labels)
    .fillna(0)
)

print("\nREGIME FREQUENCY")
print("-" * 70)

for regime, count in regime_counts.items():

    print(
        f"{regime}: "
        f"{int(count):,} days "
        f"({count / len(df) * 100:.2f}%)"
    )

# ------------------------------------------------------------
# 9. MOST LIKELY NEXT REGIME
# ------------------------------------------------------------
latest_state = df["State"].iloc[-1]

next_probabilities = (
    transition[latest_state]
)

next_regime = state_mapping[
    np.argmax(next_probabilities)
]

print("\nCURRENT REGIME")
print("-" * 70)

print(
    "Current Regime:",
    state_mapping[latest_state]
)

print("\nNext-Regime Probabilities:")

for state in range(3):

    print(
        f"{state_mapping[state]}: "
        f"{next_probabilities[state] * 100:.2f}%"
    )

print(
    "\nMost Probable Next Regime:",
    next_regime
)

# ------------------------------------------------------------
# 10. MULTI-STEP REGIME FORECAST
# ------------------------------------------------------------
print("\nMULTI-STEP REGIME FORECAST")
print("-" * 70)

current_distribution = np.zeros(3)

current_distribution[
    latest_state
] = 1

for day in [1, 5, 10, 20]:

    future_probability = (
        current_distribution @
        np.linalg.matrix_power(
            transition,
            day
        )
    )

    print(
        f"\nAfter {day} trading day(s):"
    )

    for state in range(3):

        print(
            f"{state_mapping[state]}: "
            f"{future_probability[state] * 100:.2f}%"
        )

# ------------------------------------------------------------
# 11. OBSERVED REGIME TRANSITIONS
# ------------------------------------------------------------
df["Previous_Regime"] = (
    df["Regime"].shift(1)
)

transitions = (
    df.dropna(
        subset=["Previous_Regime"]
    )
    .groupby(
        [
            "Previous_Regime",
            "Regime"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

transitions = transitions.reindex(
    index=labels,
    columns=labels,
    fill_value=0
)

print("\nOBSERVED REGIME TRANSITIONS")
print("-" * 70)

print(
    transitions
    .to_string()
)

# ------------------------------------------------------------
# 12. REGIME DURATION FROM ACTUAL DATA
# ------------------------------------------------------------

regime_change = (
    df["Regime"] !=
    df["Regime"].shift(1)
)

episode_id = (
    regime_change.cumsum()
)

# Create a separate dataframe to avoid
# duplicate "Regime" column during reset_index()
duration_data = pd.DataFrame({
    "Regime": df["Regime"].values,
    "Episode": episode_id.values
})

durations = (
    duration_data
    .groupby(
        ["Regime", "Episode"]
    )
    .size()
    .rename("Duration")
    .reset_index()
)

duration_summary = (
    durations
    .groupby("Regime")["Duration"]
    .agg(
        Episodes="count",
        Average_Duration="mean",
        Median_Duration="median",
        Maximum_Duration="max"
    )
    .reindex(labels)
)

print("\nOBSERVED REGIME DURATION")
print("-" * 70)

print(
    duration_summary
    .round(2)
    .to_string()
)

# ------------------------------------------------------------
# 13. TRANSITION MATRIX HEATMAP
# ------------------------------------------------------------
plt.figure(
    figsize=(8, 6)
)

plt.imshow(
    transition_df.values,
    aspect="auto"
)

plt.xticks(
    range(3),
    labels
)

plt.yticks(
    range(3),
    labels
)

plt.xlabel(
    "Next Regime"
)

plt.ylabel(
    "Current Regime"
)

plt.title(
    "HMM Regime Transition Probability Matrix"
)

plt.colorbar(
    label="Probability"
)

for i in range(3):

    for j in range(3):

        plt.text(
            j,
            i,
            f"{transition_df.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.show()

# ------------------------------------------------------------
# 14. REGIME TIMELINE
# ------------------------------------------------------------
numeric_regime = (
    df["Regime"]
    .map({
        "Bear": -1,
        "Neutral": 0,
        "Bull": 1
    })
)

plt.figure(
    figsize=(15, 5)
)

plt.plot(
    df.index,
    numeric_regime,
    linewidth=0.8
)

plt.yticks(
    [-1, 0, 1],
    [
        "Bear",
        "Neutral",
        "Bull"
    ]
)

plt.title(
    "NIFTY 50 — HMM Regime Timeline"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Regime"
)

plt.grid(
    alpha=0.25
)

plt.show()

# ------------------------------------------------------------
# 15. SAVE RESULTS
# ------------------------------------------------------------
df.to_csv(
    "NIFTY50_HMM_Regime_Transition_Analysis.csv"
)

transition_df.to_csv(
    "HMM_Transition_Matrix.csv"
)

persistence_df.to_csv(
    "HMM_Regime_Persistence.csv",
    index=False
)

duration_summary.to_csv(
    "HMM_Regime_Duration.csv"
)

print("\nFiles saved:")
print(
    "• NIFTY50_HMM_Regime_Transition_Analysis.csv"
)
print(
    "• HMM_Transition_Matrix.csv"
)
print(
    "• HMM_Regime_Persistence.csv"
)
print(
    "• HMM_Regime_Duration.csv"
)